In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

In [4]:
load_dotenv()

True

In [5]:
access_key = os.getenv("Access_key_ID")
secret_key = os.getenv("Secret_access_key")
bucket = os.getenv("BUCKET_NAME")
region = os.getenv("REGION_NAME")

In [6]:
spark = (
    SparkSession.builder.appName("S3DataTransformation")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.access.key", access_key)
    .config("spark.hadoop.fs.s3a.secret.key", secret_key)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .config("spark.hadoop.fs.s3a.region", region)
    .getOrCreate()
)

25/04/17 09:29:21 WARN Utils: Your hostname, brempong-HP-EliteBook-840-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.36.43 instead (on interface wlp0s20f3)
25/04/17 09:29:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/brempong/Ecommerce-DataLakehouse/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brempong/.ivy2/cache
The jars for the packages stored in: /home/brempong/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-42890500-12e4-49a9-8c0e-55daf0017650;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 280ms :: artifacts dl 11ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.1 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------

In [7]:
folder_path = f"s3a://{bucket}/raw-data/order_items_apr_2025/"
df = spark.read.csv(folder_path, header=True, inferSchema=True)
df.show()

25/04/17 09:29:37 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+-----+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|reordered|    order_timestamp|      date|
+-----+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+
|10912|   12000|   6024|                    16|       380|                1|        0|2025-04-05 04:37:00|2025-04-05|
|10913|   12000|   6024|                    15|       469|                2|        0|2025-04-05 04:37:00|2025-04-05|
|10914|   12000|   6024|                    21|       205|                3|        0|2025-04-05 04:37:00|2025-04-05|
|10915|   12000|   6024|                    27|       284|                4|        1|2025-04-05 04:37:00|2025-04-05|
|10916|   12001|   2459|                    22|       184|                1|        1|2025-04-05 13:00:00|2025-04-05|
|10917|   12001|   2459|                    16|       76

In [8]:
spark

In [9]:
df.select("date").distinct().show()

+----------+
|      date|
+----------+
|2025-04-05|
|2025-04-14|
|2025-04-07|
|2025-04-08|
|2025-04-11|
|2025-04-09|
|2025-04-13|
|2025-04-01|
|2025-04-02|
|2025-04-03|
|2025-04-12|
|2025-04-15|
|2025-04-04|
|2025-04-06|
|2025-04-10|
+----------+



In [10]:
# check for missing values
df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+---+--------+-------+----------------------+----------+-----------------+---------+---------------+----+
| id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|reordered|order_timestamp|date|
+---+--------+-------+----------------------+----------+-----------------+---------+---------------+----+
|  0|       0|      0|                     0|         0|                0|        0|              0|   0|
+---+--------+-------+----------------------+----------+-----------------+---------+---------------+----+



In [11]:
df = df.withColumn("order_time", date_format(col("order_timestamp"), "HH:mm:ss"))
df.show()

+-----+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|reordered|    order_timestamp|      date|order_time|
+-----+--------+-------+----------------------+----------+-----------------+---------+-------------------+----------+----------+
|10912|   12000|   6024|                    16|       380|                1|        0|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10913|   12000|   6024|                    15|       469|                2|        0|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10914|   12000|   6024|                    21|       205|                3|        0|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10915|   12000|   6024|                    27|       284|                4|        1|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10916|   12001|   2459|                    22|       184|                1|        1|2025-04-05 

In [12]:
df = df.withColumn(
    "reordered", when(col("reordered") == 1, "Reorder").otherwise("Not_Reorder")
)
df.show()

+-----+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|  reordered|    order_timestamp|      date|order_time|
+-----+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
|10912|   12000|   6024|                    16|       380|                1|Not_Reorder|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10913|   12000|   6024|                    15|       469|                2|Not_Reorder|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10914|   12000|   6024|                    21|       205|                3|Not_Reorder|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10915|   12000|   6024|                    27|       284|                4|    Reorder|2025-04-05 04:37:00|2025-04-05|  04:37:00|
|10916|   12001|   2459|                    22|       184|                1|    Reo

In [13]:
partitioned_df = df.repartition(15, col("date")).sortWithinPartitions(
    col("order_time"), col("reordered")
)
partitioned_df.show()

+-----+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
|   id|order_id|user_id|days_since_prior_order|product_id|add_to_cart_order|  reordered|    order_timestamp|      date|order_time|
+-----+--------+-------+----------------------+----------+-----------------+-----------+-------------------+----------+----------+
|16483|   13022|   3939|                    27|       919|                1|    Reorder|2025-04-07 00:12:00|2025-04-07|  00:12:00|
|16743|   13073|   2234|                    13|       860|                1|Not_Reorder|2025-04-07 00:14:00|2025-04-07|  00:14:00|
|16747|   13073|   2234|                    16|        54|                5|Not_Reorder|2025-04-07 00:14:00|2025-04-07|  00:14:00|
|16744|   13073|   2234|                     6|       979|                2|    Reorder|2025-04-07 00:14:00|2025-04-07|  00:14:00|
|16745|   13073|   2234|                    19|       535|                3|    Reo

In [14]:
df

DataFrame[id: int, order_id: int, user_id: int, days_since_prior_order: int, product_id: int, add_to_cart_order: int, reordered: string, order_timestamp: timestamp, date: date, order_time: string]